In [1]:
import sys

from src.data.load_cifar100 import get_cifar100_loaders, create_and_load_subset

sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd
import os
import datetime
from src.models.masked_autoencoder import MAE
from src.data.load_cifar10 import get_cifar10_loaders, create_and_load_subset_c10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [2]:
# Dane
# train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)
_, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset(
    num_classes=2,
    batch_size=64,
    selected_classes = [60, 66],
)
print(f"Trenowanie na klasach: {selected_classes}")

Używam podanych klas: [60, 66]
Trenowanie na klasach: [60, 66]


In [3]:
# Model
model = MAE().to(device)
# criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# Trening
BASE_DIR = os.getcwd()
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar10')
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar100', f'{len(selected_classes)}_classes')
print(save_dir)
# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar10')
writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar100', f'{len(selected_classes)}_classes')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(os.path.join(writer_dir, f'mae_{timestamp}'))

num_epochs = 250

train_losses = []
val_losses = []
epoch_number = 0
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)
        
        reconstructed, x_masked, mask = model(image)
        loss = model.compute_loss(image, reconstructed, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")
            
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    writer.add_scalar('Loss/train', train_loss, epoch)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)

            reconstructed, x_masked, mask = model(image)
            loss = model.compute_loss(image, reconstructed, mask)
            val_loss += loss.item()
            
        if epoch % 5 == 0:  
            n_images = min(8, image.size(0))
            
            # Oryginalne obrazy
            writer.add_images('Original', image[:n_images], epoch)
            
            # Zamaskowane obrazy
            writer.add_images('Masked', x_masked[:n_images], epoch)
            
            # Rekonstrukcje
            writer.add_images('Reconstructed', reconstructed[:n_images], epoch)
            
            # Wizualizacja maski (powtórzona dla 3 kanałów)
            mask_vis = mask[:n_images].repeat(1, 3, 1, 1)
            writer.add_images('Mask', mask_vis, epoch)
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    writer.add_scalar('Loss/val', val_loss, epoch)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")


    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'latent_dim': 256,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'selected_classes': selected_classes,
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        # checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar10_best_{timestamp}.pt')
        checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar100_best_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)


df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar10_training_results_{timestamp}.csv')
history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar100_training_results_{timestamp}_.csv')
df.to_csv(history_csv, index=False)


C:\Users\martu\Desktop\studia\magisterka\2sem\ml_projekt\src\notebooks_test_train\..\training_results\masked_autoencoder\cifar100\2_classes
  [1/250] Batch 0/13 Loss: 0.2385
Epoch [1/250]
Train Loss: 0.1392
Val Loss:   0.2177
  [2/250] Batch 0/13 Loss: 0.1032
Epoch [2/250]
Train Loss: 0.1016
Val Loss:   0.3476
  [3/250] Batch 0/13 Loss: 0.0987
Epoch [3/250]
Train Loss: 0.0947
Val Loss:   0.2085
  [4/250] Batch 0/13 Loss: 0.0950
Epoch [4/250]
Train Loss: 0.0895
Val Loss:   0.1065
  [5/250] Batch 0/13 Loss: 0.0816
Epoch [5/250]
Train Loss: 0.0865
Val Loss:   0.1001
  [6/250] Batch 0/13 Loss: 0.0855
Epoch [6/250]
Train Loss: 0.0844
Val Loss:   0.0854
  [7/250] Batch 0/13 Loss: 0.0849
Epoch [7/250]
Train Loss: 0.0812
Val Loss:   0.0831
  [8/250] Batch 0/13 Loss: 0.0741
Epoch [8/250]
Train Loss: 0.0796
Val Loss:   0.0918
  [9/250] Batch 0/13 Loss: 0.0718
Epoch [9/250]
Train Loss: 0.0790
Val Loss:   0.0859
  [10/250] Batch 0/13 Loss: 0.0790
Epoch [10/250]
Train Loss: 0.0785
Val Loss:   0.078

In [42]:
save_dir = os.path.join(os.getcwd(), '..', 'training_results', 'masked_autoencoder', 'cifar100', '50_classes')
best_checkpoint = torch.load(os.path.join(save_dir, 'masked_autoencoder_cifar100_best_20260114_013421.pt'))

saved_classes = best_checkpoint['selected_classes']
_, _, _, _, test_loader = create_and_load_subset(
    selected_classes=saved_classes,
    num_classes=50, 
    batch_size=64
)
model.load_state_dict(best_checkpoint['model_state_dict'])
correct_test = 0
total_test = 0
test_loss = 0

model.eval()
print("\nCalculating metrics on test set...")
with torch.no_grad():
    for image, _ in test_loader:
        image = image.to(device)

        reconstructed, x_masked, mask = model(image)
        loss = model.compute_loss(image, reconstructed, mask)
        test_loss += loss.item()

test_loss /= len(test_loader)
# writer.add_scalar('Loss/test', test_loss)
print("FINAL EVALUATION RESULTS")
print(f"Test Loss:           {test_loss:.6f}")      
        
    

Używam podanych klas: [4, 87, 50, 6, 99, 66, 74, 58, 83, 61, 28, 68, 31, 67, 14, 64, 95, 39, 1, 47, 71, 22, 54, 59, 53, 19, 25, 72, 60, 69, 43, 0, 76, 78, 45, 89, 17, 7, 11, 46, 15, 52, 51, 18, 82, 98, 49, 63, 70, 65]

Calculating metrics on test set...
FINAL EVALUATION RESULTS
Test Loss:           0.060132
